# 13 - M&A Screener
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/13-ma-screener/ma_screener_workbook.ipynb)

Score acquisition targets against a multi-criteria rubric and produce a ranked shortlist.

**What you will build:** A single-call agent that reads an acquirer brief + target list and returns a `ScreeningResult` with each target scored on three dimensions (0-10 each), a threshold gate per dimension, and a ranked shortlist.

**Harness focus:** Multi-criteria financial scoring rubric + grounded shortlist

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: Why multi-criteria beats a single score

A single composite score hides which dimension failed. If a target has excellent financials but is in the wrong geography, a single score of 6/10 looks acceptable -- but you would never proceed.

The solution: score each dimension independently AND apply a minimum threshold per dimension. A target that fails any threshold is screened out regardless of its composite score.

| Dimension | Measures | Threshold |
|-----------|----------|-----------|
| strategic_fit | Sector, geography, business model | >= 5 |
| financial_fit | Revenue size, EBITDA margin, growth | >= 5 |
| operational_fit | Integration complexity, culture | >= 5 |

**overall_score** = sum of three dimensions (max 30)

**recommendation**: proceed (>=20), monitor (14-19), pass (<14 or any threshold miss)

## Part 2: Schema -- DimensionScore as a nested model

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class DimensionScore(BaseModel):
    score: int = Field(description="0-10 score for this dimension")
    rationale: str = Field(description="Evidence-based reason for the score")
    meets_threshold: bool = Field(description="True if this dimension clears the minimum bar")

class TargetAssessmentCard(BaseModel):
    company_name: str
    sector: str
    geography: str
    strategic_fit: DimensionScore
    financial_fit: DimensionScore
    operational_fit: DimensionScore
    overall_score: int = Field(description="Weighted composite 0-30")
    recommendation: Literal["proceed", "monitor", "pass"]
    investment_thesis: str
    key_risks: List[str]
    suggested_next_step: str

class ScreeningResult(BaseModel):
    acquirer: Optional[str] = None
    rubric_summary: str
    shortlist: List[TargetAssessmentCard]
    screened_out: List[str]
    recommendation: str

print("Schema defined.")

## Part 3: System prompt -- explicit rubric in the prompt

The system prompt encodes the scoring rules, threshold definition, and ranking instruction explicitly. The model must follow them exactly.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

SCREENER_SYSTEM = SystemMessage("""
You are a senior M&A analyst at a private equity firm.

Score each target across three dimensions (0-10 each):
  - strategic_fit   : sector alignment, geography, business model match
  - financial_fit   : revenue size, EBITDA margin, growth vs. the stated rubric
  - operational_fit : integration complexity, management quality, cultural alignment

Threshold: a score below 5 on ANY dimension is a fail -- that target is screened out.

overall_score = strategic_fit + financial_fit + operational_fit  (max 30)
recommendation: proceed (>=20), monitor (14-19), pass (<14 or any threshold miss)

Rank the shortlist by overall_score descending.
Set meets_threshold = True only when the dimension score >= 5.""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
screener = SCREENER_SYSTEM | llm.with_structured_output(ScreeningResult)
print("Agent ready.")

## Part 4: Run on 5 synthetic targets

In [ ]:
BRIEF = """
ACQUIRER: Apex Capital Partners
B2B SaaS bolt-on acquisitions. Criteria: GBP 10-50m revenue, 15%+ EBITDA, UK/EU only.

TARGETS:
1. FieldSense Ltd -- UK, field service SaaS, GBP 18m rev, 22% EBITDA, +31% growth
2. LogiFlow GmbH -- Germany, logistics SaaS, GBP 12m rev, 8% EBITDA, 60% single customer
3. RetailAI Ltd -- UK, retail tech, GBP 45m rev, -3% EBITDA (loss-making)
4. CloudOps Nordic AB -- Sweden, infra SaaS, GBP 22m rev, 19% EBITDA, +24% growth
5. DataBridge Inc -- USA, data SaaS, GBP 35m rev, 24% EBITDA (wrong geography)
"""

result = screener.invoke(HumanMessage(content="Screening brief and targets:\n\n" + BRIEF))
print(f"Shortlist: {len(result.shortlist)} | Screened out: {len(result.screened_out)}")

## Part 5: Print the ranked shortlist

In [ ]:
print(f"Rubric: {result.rubric_summary}")
print(f"Top-line: {result.recommendation}\n")
for t in result.shortlist:
    print(f"[{t.overall_score}/30] {t.company_name} -- {t.recommendation.upper()}")
    print(f"  Strategic: {t.strategic_fit.score}/10 | "
          f"Financial: {t.financial_fit.score}/10 | "
          f"Operational: {t.operational_fit.score}/10")
    print(f"  Thesis: {t.investment_thesis}")
    print()
print(f"Screened out: {', '.join(result.screened_out)}")

## Exercise: Add a fourth dimension -- management_quality

Extend the schema with a `management_quality` DimensionScore and update the system prompt to score it (criteria: team depth, CEO track record, succession planning).

The overall_score should now be out of 40.

**Starter:** Add the field to TargetAssessmentCard and re-run.

In [ ]:
# Starter: extend the schema with management_quality
class TargetAssessmentCardV2(BaseModel):
    company_name: str
    sector: str
    geography: str
    strategic_fit: DimensionScore
    financial_fit: DimensionScore
    operational_fit: DimensionScore
    management_quality: DimensionScore  # NEW dimension
    overall_score: int = Field(description="Composite 0-40 (four dimensions)")
    recommendation: Literal["proceed", "monitor", "pass"]
    investment_thesis: str
    key_risks: List[str]
    suggested_next_step: str

class ScreeningResultV2(BaseModel):
    acquirer: Optional[str] = None
    rubric_summary: str
    shortlist: List[TargetAssessmentCardV2]
    screened_out: List[str]
    recommendation: str

# TODO: update SCREENER_SYSTEM to include management_quality scoring rules
# TODO: rebuild screener_v2 = updated_system | llm.with_structured_output(ScreeningResultV2)
# TODO: run screener_v2 on BRIEF and print the new scores

In [ ]:
# Answer key: updated system prompt + re-run
SCREENER_SYSTEM_V2 = SystemMessage("""
You are a senior M&A analyst. Score each target on FOUR dimensions (0-10 each):
  - strategic_fit      : sector, geography, business model
  - financial_fit      : revenue, EBITDA, growth
  - operational_fit    : integration complexity, culture
  - management_quality : CEO track record, team depth, succession planning

Threshold: below 5 on ANY dimension = screened out.
overall_score = sum of four dimensions (max 40).
recommend proceed if >=26, monitor if 18-25, pass otherwise.
Rank shortlist by overall_score descending.""")

screener_v2 = SCREENER_SYSTEM_V2 | llm.with_structured_output(ScreeningResultV2)
result_v2 = screener_v2.invoke(HumanMessage(content="Screening brief and targets:\n\n" + BRIEF))
for t in result_v2.shortlist:
    print(f"[{t.overall_score}/40] {t.company_name} -- mgmt: {t.management_quality.score}/10")

## What You Built

- A multi-criteria screener that applies a per-dimension threshold gate
- `DimensionScore` as a reusable nested model (score + rationale + meets_threshold)
- A system prompt that encodes the rubric, threshold, and ranking rules explicitly
- The schema enforces separation between shortlist and screened_out

**Next steps:** Try example 14 (fan-out output) or adapt this screener to a job candidate / supplier qualification use case.